# Account C — Audit / Shadow Eval (FIXED: embedded baseline engine)

This version writes the frozen baseline engine automatically, so `baseline=None` is fixed. Phase-1 path is hard-coded to your Kaggle dataset path. Candidate is optional; if `None`, identity audit is run.

In [ ]:
from pathlib import Path
import json, os

OUT_DIR=Path("/kaggle/working/audit")
OUT_DIR.mkdir(parents=True,exist_ok=True)
PHASE1_PATH=Path("/kaggle/input/datasets/nguyenminhtric/test-pipeline/exact_eval_round1_Astatine.json")
CANDIDATE_ENGINE=None  # set to /path/to/candidate.py if auditing a candidate
print("PHASE1_PATH =", PHASE1_PATH)
print("exists:", PHASE1_PATH.exists())
if not PHASE1_PATH.exists():
    raise FileNotFoundError(f"Phase-1 log not found at {PHASE1_PATH}")
print("OUT_DIR =", OUT_DIR)

In [ ]:
BASELINE_ENGINE=Path("/kaggle/working/v40_4_entity_conjunctive_engine.py")
BASELINE_ENGINE.write_text('\n# -*- coding: utf-8 -*-\n"""v40.4/v40.5 MC-only certificate engine — frozen baseline.\nEntity-grounded conjunctive Horn solver, abstain-safe.\n"""\nimport re, json, os, sys\n\nSTOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'its\',\'it\',\'they\',\'them\',\n      \'is\',\'are\',\'was\',\'were\',\'be\',\'been\',\'has\',\'have\',\'had\',\'then\',\'if\',\'no\',\'not\',\'with\',\'as\',\'by\',\'from\',\n      \'artifact\',\'package\',\'manuscript\',\'sample\',\'batch\',\'item\',\'device\',\'record\',\'file\',\'student\'}\n\nLEMMA = {\n    "approved":"approve","approval":"approve","approves":"approve","approve":"approve",\n    "required":"require","requires":"require","requirement":"require","requirements":"require","require":"require",\n    "recommended":"recommend","recommendation":"recommend","recommendations":"recommend","recommends":"recommend","recommend":"recommend",\n    "administered":"administer","administration":"administer","administers":"administer","administer":"administer",\n    "verified":"verify","verification":"verify","verifies":"verify","verify":"verify",\n    "certified":"certify","certification":"certify","certifies":"certify","certify":"certify",\n    "assigned":"assign","assignment":"assign","assigns":"assign","assign":"assign",\n    "eligible":"eligible","eligibility":"eligible",\n    "authorized":"authorize","authorization":"authorize","authorizes":"authorize","authorize":"authorize",\n    "reviewed":"review","reviews":"review","review":"review",\n}\n\ndef _stem(t):\n    t = t.lower()\n    if t in LEMMA: return LEMMA[t]\n    if re.search(r\'(ss|us|is)$\',t): return t\n    if re.search(r\'(ches|shes|xes|zes|ses)$\',t): return t[:-2]\n    if re.search(r\'ies$\',t): return t[:-3]+\'y\'\n    if t.endswith(\'s\'): t=t[:-1]\n    return re.sub(r\'(ing|ed)$\',\'\',t)\n\ndef atom_key(phrase):\n    s=re.sub(r\'(?<!^)(?=[A-Z])\',\' \',str(phrase)).lower()\n    nums=re.findall(r\'\\d+\', s)\n    toks=[ _stem(w) for w in re.findall(r\'[a-zA-Z]+\', s) ]\n    toks=[t for t in toks if t and t not in STOP and len(t)>2]\n    keys=sorted(set(toks)) + ["N"+n for n in sorted(set(nums))]\n    return "".join(w.capitalize() for w in keys) if keys else None\n\n_LEAD=re.compile(r"^\\s*(if|then|that|who|which|it|its|their|this)\\b",re.I)\n_VERB=re.compile(r"\\b(cannot|can not|can|could|may|might|must|should|shall|will|would|is not|are not|was not|were not|isn\'t|aren\'t|is|are|was|were|has no|have no|had no|has|have|had|lacks?|without|requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|assigned|be|been|being)\\b",re.I)\nACTION_VERBS={\'receives\',\'receive\',\'provides\',\'provide\',\'shows\',\'show\',\'states\',\'state\',\'monitors\',\'monitor\',\'captures\',\'capture\',\'enters\',\'enter\',\'requires\',\'require\',\'needs\',\'need\',\'gains\',\'gain\',\'completed\',\'complete\',\'contains\',\'contain\',\'reports\',\'report\',\'releases\',\'release\',\'passes\',\'pass\',\'improves\',\'improve\',\'supports\',\'support\',\'recommends\',\'recommend\',\'administered\',\'administer\',\'approved\',\'approve\'}\n\ndef to_literal(clause):\n    c=clause.strip().rstrip(\'.\').strip()\n    c=_LEAD.sub(\'\',c).strip()\n    neg=bool(re.search(r"\\b(no|not|cannot|can not|never|lacks?|without|isn\'t|aren\'t|incomplete|missing|lacking|nor|un(?:able|verified|established))\\b",c,re.I))\n    m=_VERB.search(c)\n    pred=c[m.end():].strip() if m else c\n    verb=(m.group(1).lower() if m else \'\')\n    if m and verb in ACTION_VERBS:\n        pred = verb + \' \' + pred\n    for _ in range(4):\n        pred=re.sub(r"^\\s*(be|been|being|to|a|an|the|no|not|its|their)\\b","",pred,flags=re.I).strip()\n    a=atom_key(pred)\n    return (a,neg) if a else None\n\ndef _consequent_guard(cons_atom, all_text):\n    if not cons_atom: return False\n    toks = set(re.findall(r\'[A-Z][a-z0-9]*\', cons_atom))\n    raw = " ".join(all_text).lower()\n    hit = sum(1 for t in toks if t.lower() in raw)\n    return hit >= max(1, min(2, len(toks)))\n\ndef parse_premise(p, all_text=None):\n    s=str(p).strip()\n    m=re.search(r\'^\\s*if\\b(.+?),?\\s*\\bthen\\b(.+)$\',s,re.I)\n    if m:\n        ante=re.split(r\'\\band\\b\',m.group(1),flags=re.I)\n        lits=[to_literal(x) for x in ante]; lits=[l for l in lits if l]\n        con=to_literal(m.group(2))\n        if con and lits: return (\'rule\',lits,con)\n        return None\n    m2=re.search(r\'^\\s*(every|all)\\s+[a-zA-Z]+s?\\s+(.+?)\\s+\\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are)\\b\\s+(.+)$\',s,re.I)\n    if m2:\n        cond=m2.group(2).strip()\n        cons=(m2.group(3)+" "+m2.group(4)).strip()\n        litc=to_literal(cond); litd=to_literal(cons)\n        if litc and litd:\n            if all_text is None or _consequent_guard(litd[0], all_text):\n                return (\'rule\',[litc],litd)\n            return None\n    if re.search(r\'^\\s*(no premise|it (is|cannot)|unknown|there is no information)\',s,re.I): return None\n    lit=to_literal(s)\n    return (\'fact\',lit) if lit else None\n\ndef solve_entity(premises, all_text=None):\n    facts={}; rules=[]\n    all_text = list(all_text or []) + list(premises or [])\n    for i,p in enumerate(premises):\n        pp=parse_premise(p, all_text=all_text)\n        if not pp: continue\n        if pp[0]==\'fact\':\n            a,neg=pp[1]; facts.setdefault(a,(not neg,[i]))\n        else:\n            rules.append((i,pp[1],pp[2]))\n    changed=True\n    while changed:\n        changed=False\n        for i,lits,con in rules:\n            ca,cneg=con\n            ok=True; path=[i]\n            for a,neg in lits:\n                if a in facts and facts[a][0]==(not neg): path += facts[a][1]\n                else: ok=False; break\n            if ok and ca not in facts:\n                facts[ca]=((not cneg),sorted(set(path))); changed=True\n    return facts\n\n_META_RE=re.compile(r"\\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\\b",re.I)\n\ndef decompose_option(opt):\n    t=re.sub(r\'^\\s*[A-Da-d][.):]\\s*\',\'\',str(opt)).strip()\n    t=re.split(r\'\\bbecause\\b\', t, maxsplit=1, flags=re.I)[0].strip()\n    parts=re.split(r\',\\s*but\\s+|\\s+but\\s+|;\\s+|\\s+while\\s+|\\s+whereas\\s+|\\s+and\\s+\',t,flags=re.I)\n    claims=[]\n    for p in parts:\n        p=p.strip()\n        if not p: continue\n        is_meta=bool(_META_RE.search(p))\n        lit=to_literal(p)\n        claims.append((lit,is_meta,p))\n    return claims\n\ndef answer_mc(premises, options):\n    all_text = list(premises or []) + list(options or [])\n    facts=solve_entity(premises, all_text=all_text)\n    res={}\n    for lab,opt in zip("ABCD",options):\n        claims=decompose_option(opt)\n        if not claims: res[lab]=(\'UNSUP\',[]); continue\n        status=\'PROVEN\'; path=[]\n        for lit,is_meta,txt in claims:\n            if lit is None: status=\'UNSUP\'; break\n            a,neg=lit; have=a in facts\n            val=facts[a][0] if have else None\n            if is_meta:\n                if have and val==True: status=\'DISPROVEN\'; break\n            else:\n                if have and val==(not neg): path += facts[a][1]\n                elif have and val==(neg): status=\'DISPROVEN\'; break\n                else: status=\'UNSUP\'; break\n        res[lab]=(status, sorted(set(path)))\n    proven=[l for l in res if res[l][0]==\'PROVEN\']\n    if len(proven)==1: return proven[0],res[proven[0]][1],\'entity_unique_proof\',res\n    return None,[],(\'multiple\' if proven else \'none\'),res\n',encoding="utf-8")
print("Wrote baseline:", BASELINE_ENGINE, "size", BASELINE_ENGINE.stat().st_size)

In [ ]:
SHADOW_EVAL=Path("/kaggle/working/shadow_eval.py")
SHADOW_EVAL.write_text('\nimport json,re,sys,os\n\ndef load_engine(path):\n    src=open(path).read()\n    src=src.split("# ================= Phase-1 REPLAY")[0] if "# ================= Phase-1 REPLAY" in src else src.split("if __name__")[0]\n    g={}; exec(src,g); return g\n\ndef opt_texts(rp):\n    q=rp.get("query","")\n    f=[o[1].strip().replace("\\n"," ") for o in re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)",q,flags=re.S)]\n    return f if len(f)>=2 else []\n\ndef run_engine(g,t1):\n    out={}\n    for l in t1:\n        rp=l["request_payload"]; opts=opt_texts(rp)\n        if not opts:\n            out[l["query_id"]]={"answer":None,"premises_used":[],"why":"no_options","res":{}}\n            continue\n        a,pu,why,res=g["answer_mc"](rp.get("premises",[]) or [], opts)\n        out[l["query_id"]]={"answer":a,"premises_used":sorted(pu),"why":why,"res":{k:res[k][0] for k in res}}\n    return out\n\ndef taxonomy(g,l,r):\n    if r["answer"] is not None: return "FIRED"\n    if r["why"]=="multiple": return "multiple_direct_proven"\n    prems=l["request_payload"].get("premises",[]) or []\n    facts=rules=0\n    for p in prems:\n        try: pp=g["parse_premise"](p)\n        except TypeError: pp=g["parse_premise"](p,all_text=prems)\n        if not pp: continue\n        facts += (pp[0]=="fact"); rules += (pp[0]=="rule")\n    if facts==0 and rules==0: return "entity_conjunction_parse_miss"\n    opts=opt_texts(l["request_payload"])\n    if not opts: return "ynu_unsupported"\n    if any(re.search(r",?\\s*(but|and|;|while|whereas|because)\\s+",o,re.I) for o in opts): return "compound_option_unparsed"\n    return "predicate_mismatch"\n\ndef evaluate(baseline_path,candidate_path,phase1_path,outdir):\n    os.makedirs(outdir,exist_ok=True)\n    d=json.load(open(phase1_path)); t1=[l for l in d["logs"] if l.get("type")=="type1"]\n    exp={l["query_id"]:(l.get("expected") or {}) for l in t1}\n    gold={q:str(e.get("answer","")).strip() for q,e in exp.items()}\n    gB=load_engine(baseline_path); B=run_engine(gB,t1)\n    gC=load_engine(candidate_path); C=run_engine(gC,t1)\n    base_fired={q for q,r in B.items() if r["answer"] is not None}\n    cand_fired={q for q,r in C.items() if r["answer"] is not None}\n    def correct(q,r):\n        g=gold[q].upper(); a=str(r["answer"]).strip()\n        return a.upper()==g if g in {"A","B","C","D"} else a.lower()==g.lower()\n    added=sorted(cand_fired-base_fired)\n    lost=sorted(base_fired-cand_fired)\n    changed_premises=[q for q in (base_fired & cand_fired) if B[q]["premises_used"]!=C[q]["premises_used"]]\n    wrong=sorted([q for q in cand_fired if not correct(q,C[q])])\n    new_multi=[q for q,r in C.items() if r["why"]=="multiple" and B[q]["why"]!="multiple"]\n    if wrong or lost or changed_premises or new_multi: decision="REJECT"\n    elif added: decision="PROMOTE"\n    else: decision="ACCEPT_SHADOW_ONLY"\n    diff={"candidate":os.path.basename(candidate_path),"baseline":os.path.basename(baseline_path),"n":len(t1),\n          "baseline_fired_set":sorted(base_fired),"candidate_fired_set":sorted(cand_fired),"added":added,"lost":lost,\n          "changed_premises":changed_premises,"candidate_wrong":wrong,"new_multiple_proven":new_multi,"decision":decision,\n          "reject_reasons":[r for r,ok in [("has_wrong",bool(wrong)),("lost_cases",bool(lost)),("changed_premises",bool(changed_premises)),("new_multiple_proven",bool(new_multi))] if ok]}\n    json.dump(diff,open(os.path.join(outdir,"candidate_diff_report.json"),"w"),indent=1)\n    from collections import Counter\n    tax=Counter(); rows=[]\n    for l in t1:\n        t=taxonomy(gB,l,B[l["query_id"]]); tax[t]+=1\n        rows.append({"query_id":l["query_id"],"taxonomy":t,"gold":gold[l["query_id"]],"old_status":l.get("status")})\n    json.dump({"taxonomy":dict(tax),"cases":rows},open(os.path.join(outdir,"abstain_taxonomy.json"),"w"),indent=1)\n    certs=[{"query_id":q,"answer":B[q]["answer"],"premises_used":B[q]["premises_used"],"rule":B[q]["why"],"premises_used_matches_gold":(B[q]["premises_used"]==sorted(exp[q].get("premises_used") or []))} for q in sorted(base_fired)]\n    json.dump({"baseline":os.path.basename(baseline_path),"fired":len(certs),"cases":certs},open(os.path.join(outdir,"proof_certificate_cases.json"),"w"),indent=1)\n    json.dump({"decision":decision,"diff":diff,"golden_rule":"PROMOTE iff added!=[] and lost==[] and changed_premises==[] and wrong==[] and no new multiple-proven"},open(os.path.join(outdir,"safe_upgrade_decision.json"),"w"),indent=1)\n    return diff\n',encoding="utf-8")
print("Wrote shadow_eval:", SHADOW_EVAL, "size", SHADOW_EVAL.stat().st_size)

In [ ]:
if CANDIDATE_ENGINE is None:
    CANDIDATE_ENGINE=str(BASELINE_ENGINE)
else:
    CANDIDATE_ENGINE=str(CANDIDATE_ENGINE)
print("baseline:", BASELINE_ENGINE)
print("candidate:", CANDIDATE_ENGINE)
print("phase1:", PHASE1_PATH)
if not Path(CANDIDATE_ENGINE).exists():
    raise FileNotFoundError(f"Candidate engine not found: {CANDIDATE_ENGINE}")

In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location("shadow_eval", str(SHADOW_EVAL))
mod=importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
diff=mod.evaluate(str(BASELINE_ENGINE), str(CANDIDATE_ENGINE), str(PHASE1_PATH), str(OUT_DIR))
print(json.dumps(diff,indent=2))
print("\nDecision:")
print((OUT_DIR/"safe_upgrade_decision.json").read_text()[:2000])

## Outputs

- `/kaggle/working/audit/candidate_diff_report.json`
- `/kaggle/working/audit/abstain_taxonomy.json`
- `/kaggle/working/audit/proof_certificate_cases.json`
- `/kaggle/working/audit/safe_upgrade_decision.json`